# 第7章：构建聊天应用
## Microsoft Foundry Models API 快速入门

本笔记本改编自包含访问 [Azure OpenAI](notebook-azure-openai.ipynb) 服务笔记本的 [Azure OpenAI 示例仓库](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)。

> **注意：** GitHub Models 将于2026年7月底退役。该笔记本现在以提供相同免费试用模型目录和 Azure AI 推理 SDK 体验的 [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) 为目标。


# 概述  
“大型语言模型是将文本映射到文本的函数。给定一个输入文本字符串，大型语言模型尝试预测接下来会出现的文本”(1)。本“快速入门”笔记本将向用户介绍高级的LLM概念、使用AML入门的核心包依赖、一个关于提示设计的初步介绍，以及几个不同用例的简短示例。


## 目录  

[概览](#overview)  
[如何使用 OpenAI 服务](#how-to-use-openai-service)  
[1. 创建你的 OpenAI 服务](#1.-creating-your-openai-service)  
[2. 安装](#2.-installation)    
[3. 凭据](#3.-credentials)  

[使用案例](#use-cases)    
[1. 总结文本](#1.-summarize-text)  
[2. 文本分类](#2.-classify-text)  
[3. 生成新产品名称](#3.-generate-new-product-names)  
[4. 微调分类器](#4.fine-tune-a-classifier)  

[参考文献](#references)


### 构建你的第一个提示  
这个简短的练习将为你介绍如何在 Microsoft Foundry Models 中向模型提交提示，以完成一个简单的任务“摘要”。  


<strong>步骤</strong>：  
1. 如果尚未安装，请在你的 Python 环境中安装 `azure-ai-inference` 库。  
2. 加载标准辅助库并设置你的 Microsoft Foundry Models 凭据。  
3. 为你的任务选择一个模型  
4. 为模型创建一个简单的提示  
5. 向模型 API 提交你的请求！  


### 1. 安装 `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. 导入辅助库并实例化凭据


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. 寻找合适的模型  
像 GPT-4o 和 GPT-4o mini 这样的模型能够理解和生成自然语言，并且可以在 Microsoft Foundry Models 目录中找到，与 Meta、Mistral、Cohere 和 Microsoft 的模型一起提供。


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. 提示设计  

“大型语言模型的神奇之处在于，通过在大量文本上训练以最小化预测误差，模型最终学会了对这些预测有用的概念。例如，它们学会了像”(1):

* 如何拼写
* 语法是如何运作的
* 如何释义
* 如何回答问题
* 如何进行对话
* 如何用多种语言写作
* 如何编程
* 等等

#### 如何控制大型语言模型  
“在所有输入中，文本提示无疑是对大型语言模型影响最大的”(1).

可以通过几种方式提示大型语言模型生成输出：

指令：告诉模型你想要什么
补全：诱导模型补全你想要的开始部分
演示：向模型展示你想要的，通过以下任一方式：
提示中给出几个样例
在微调训练数据集中提供数百或数千个样例”



#### 创建提示的三个基本指导原则：

<strong>展示并说明</strong>。通过指令、示例或两者结合明确表达你的需求。如果你想让模型按字母顺序对一列项目排序或按情感对段落分类，就向它展示你需要它这么做。

<strong>提供高质量数据</strong>。如果你想构建分类器或让模型遵循某种模式，确保有足够的示例。务必校对你的示例——模型通常足够聪明，可以识别基本的拼写错误并给出回应，但它也可能认为这是刻意为之，这会影响回答。

**检查你的设置。** temperature 和 top_p 设置控制模型生成回答的确定性。如果你要的是唯一正确答案，应该将这些值调低。如果想获得更多样化的回答，可以调高。人们对这些设置最大的错误是认为它们是“聪明”或“创造力”的控制开关。


来源：https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### 重复相同的调用，结果如何比较？ 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## 总结文本  
#### 挑战  
通过在文本段落末尾添加“tl;dr:”来总结文本。注意模型如何理解在没有额外指令的情况下执行多种任务。您可以使用比tl;dr更具描述性的提示来改变模型的行为并自定义您获得的摘要(3)。  

最近的研究表明，通过在大量文本语料上进行预训练，然后在特定任务上进行微调，能够在许多NLP任务和基准测试中取得显著提升。虽然架构通常是任务无关的，但这种方法仍然需要数千甚至数万个特定任务的微调数据集。相比之下，人类通常只需少量示例或简单指令就能执行新的语言任务——这是当前NLP系统仍普遍难以做到的。我们展示了扩大语言模型规模极大地提升了任务无关的少样本性能，有时甚至能够与先前的最先进微调方法竞争。  



tl;dr  


# 多种用例的练习  
1. 摘要文本  
2. 分类文本  
3. 生成新产品名称


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 文本分类  
#### 挑战  
将项目分类到推理时提供的类别中。在下面的示例中，我们在提示中同时提供了类别和要分类的文本(*playground_reference)。 

客户咨询：你好，我笔记本键盘上的一个键最近坏了，我需要一个替换件：

分类类别：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 生成新产品名称
#### 挑战
根据示例词创建产品名称。这里我们在提示中包含了关于我们将为其生成名称的产品的信息。我们还提供了一个类似的例子来展示我们希望收到的模式。我们还将温度值设置得很高，以增加随机性和更具创新性的响应。

产品描述：家用奶昔机
种子词：快速，健康，紧凑。
产品名称：HomeShaker, Fit Shaker, QuickShake, Shake Maker

产品描述：一双适合任何脚型的鞋子。
种子词：适应性，合脚，全适应。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# 参考资料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [微调 GPT 模型进行文本分类的最佳实践](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 更多帮助  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# 贡献者
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
